In [ ]:
import os, io, random, gc
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
from sklearn.metrics import f1_score
import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "GPU non actif -> Settings > Accelerator > GPU"
print("device:", device, torch.cuda.get_device_name(0))


In [ ]:
BASE = Path("/kaggle/input")
comp = next(p.parent for p in BASE.rglob("sample_submission.csv"))

BACKBONE = "tf_efficientnet_b0_ns"
IMG = 384
BATCH = 48
EPOCHS = 8
LR = 3e-4
PRETRAINED = True

def load_gt(path):
    df = pd.read_csv(path)
    return df.rename(columns={df.columns[1]: "classe"})

train_df = load_gt(comp / "train" / "ground_truth.csv")
val_df = load_gt(comp / "val" / "ground_truth.csv")
sub = pd.read_csv(comp / "sample_submission.csv")
LABEL_COL = [c for c in sub.columns if c != "image_id"][0]

train_imgs = comp / "train" / "images"
val_imgs = comp / "val" / "images"
test_imgs = comp / "test" / "images"

train_lbl = dict(zip(train_df.image_id, train_df.classe))
val_y = val_df["classe"].values


In [ ]:
def ela(pil, q=90, scale=12):
    buf = io.BytesIO()
    pil.save(buf, "JPEG", quality=q); buf.seek(0)
    re = Image.open(buf).convert("RGB")
    d = np.abs(np.asarray(pil, np.int16) - np.asarray(re, np.int16))
    return np.clip(d * scale, 0, 255).astype(np.uint8)


def noise(pil, scale=4):
    a = np.asarray(pil, np.float32)
    res = a - cv2.GaussianBlur(a, (0, 0), 1.0)
    return np.clip(res * scale + 128, 0, 255).astype(np.uint8)

def forensic(pil):
    return np.concatenate([ela(pil), noise(pil)], axis=2)


In [ ]:
def build_cache(folder, ids, fn):
    def make(name):
        with Image.open(folder / name) as im:
            im = im.convert("RGB"); im.thumbnail((IMG, IMG))
            return name, fn(im)
    with ThreadPoolExecutor(max_workers=8) as ex:
        return dict(ex.map(make, list(ids)))

rgb_fn = lambda im: np.asarray(im)


class Imgs(Dataset):
    def __init__(self, ids, cache, tf, labels=None):
        self.ids = list(ids); self.cache = cache; self.tf = tf; self.labels = labels

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        name = self.ids[i]
        x = self.tf(image=self.cache[name])["image"]
        if self.labels is None:
            return x, name
        return x, torch.tensor(self.labels[name], dtype=torch.float32)


In [ ]:
def transforms(nch):
    mean, std = [0.5] * nch, [0.5] * nch
    aug = A.Compose([
        A.PadIfNeeded(IMG, IMG, border_mode=0),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.Normalize(mean, std, max_pixel_value=255.0), ToTensorV2(),
    ])
    ev = A.Compose([
        A.PadIfNeeded(IMG, IMG, border_mode=0),
        A.Normalize(mean, std, max_pixel_value=255.0), ToTensorV2(),
    ])
    return aug, ev

dl_args = dict(num_workers=4, pin_memory=True, persistent_workers=True)


In [ ]:
import copy

@torch.no_grad()
def predict(model, dl, tta=False):
    model.eval()
    out = []
    for x, _ in dl:
        x = x.to(device)
        views = [x, torch.flip(x, [3]), torch.flip(x, [2])] if tta else [x]
        with torch.amp.autocast("cuda"):
            p = sum(torch.sigmoid(model(v).squeeze(1)) for v in views) / len(views)
        out.append(p.float().cpu().numpy())
    return np.concatenate(out)


def train_one(train_cache, val_cache, tf_tr, tf_ev, in_chans, tag):
    tr_dl = DataLoader(Imgs(train_df.image_id, train_cache, tf_tr, train_lbl),
                       BATCH, shuffle=True, drop_last=True, **dl_args)
    va_dl = DataLoader(Imgs(val_df.image_id, val_cache, tf_ev), BATCH, shuffle=False, **dl_args)

    model = timm.create_model(BACKBONE, pretrained=PRETRAINED, num_classes=1, in_chans=in_chans).to(device)
    pw = (train_df.classe == 0).sum() / max((train_df.classe == 1).sum(), 1)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, LR, epochs=EPOCHS, steps_per_epoch=len(tr_dl))
    scaler = torch.amp.GradScaler("cuda")

    best_f1, best_state = 0.0, None
    for epoch in range(EPOCHS):
        model.train()
        for x, y in tr_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                loss = loss_fn(model(x).squeeze(1), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
        f1 = f1_score(val_y, (predict(model, va_dl) > 0.5).astype(int))
        if f1 > best_f1:
            best_f1, best_state = f1, copy.deepcopy(model.state_dict())
        print(f"[{tag}] epoch {epoch}  val_f1@0.5={f1:.4f}")
    model.load_state_dict(best_state)
    print(f"[{tag}] best val_f1@0.5={best_f1:.4f}")
    return model


In [ ]:
rgb_tr, rgb_ev = transforms(3)
rgb_train_cache = build_cache(train_imgs, train_df.image_id, rgb_fn)
rgb_val_cache = build_cache(val_imgs, val_df.image_id, rgb_fn)

rgb_model = train_one(rgb_train_cache, rgb_val_cache, rgb_tr, rgb_ev, 3, "rgb")

rgb_test_cache = build_cache(test_imgs, sub.image_id, rgb_fn)
val_rgb = predict(rgb_model, DataLoader(Imgs(val_df.image_id, rgb_val_cache, rgb_ev), BATCH, **dl_args), tta=True)
test_rgb = predict(rgb_model, DataLoader(Imgs(sub.image_id, rgb_test_cache, rgb_ev), BATCH, **dl_args), tta=True)

del rgb_train_cache, rgb_val_cache, rgb_test_cache, rgb_model
gc.collect(); torch.cuda.empty_cache()


In [ ]:
for_tr, for_ev = transforms(6)
for_train_cache = build_cache(train_imgs, train_df.image_id, forensic)
for_val_cache = build_cache(val_imgs, val_df.image_id, forensic)

for_model = train_one(for_train_cache, for_val_cache, for_tr, for_ev, 6, "forensic")

for_test_cache = build_cache(test_imgs, sub.image_id, forensic)
val_for = predict(for_model, DataLoader(Imgs(val_df.image_id, for_val_cache, for_ev), BATCH, **dl_args), tta=True)
test_for = predict(for_model, DataLoader(Imgs(sub.image_id, for_test_cache, for_ev), BATCH, **dl_args), tta=True)

del for_train_cache, for_val_cache, for_test_cache, for_model
gc.collect(); torch.cuda.empty_cache()


In [ ]:
for name, vp in [("rgb", val_rgb), ("forensic", val_for), ("ensemble", (val_rgb + val_for) / 2)]:
    print(f"{name:9} val_f1@0.5={f1_score(val_y, (vp > 0.5).astype(int)):.4f}")

val_ens = (val_rgb + val_for) / 2
grid = np.linspace(0.05, 0.95, 181)
scores = [f1_score(val_y, (val_ens > t).astype(int)) for t in grid]
best_t = grid[int(np.argmax(scores))]
print(f"seuil optimal={best_t:.3f}  f1_ensemble={max(scores):.4f}")


In [ ]:
test_ens = (test_rgb + test_for) / 2
out = sub[["image_id"]].copy()
out[LABEL_COL] = (test_ens > best_t).astype(int)
assert out[LABEL_COL].isin([0, 1]).all() and out.image_id.is_unique
out.to_csv("submission.csv", index=False)
out[LABEL_COL].value_counts()
